# Partie 4 — Analyse des tendances & Scoring
**Entrée** : `immobilier_predictions.csv`  
**Sortie** : `immobilier_scored.csv`

## 1. Installation & Imports

In [1]:
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','pyspark','--quiet'])
print('OK')

OK



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import os, shutil, re, subprocess, time
import pandas as pd

subprocess.run('pkill -f SparkSubmit',   shell=True, capture_output=True)
subprocess.run('pkill -f pyspark',       shell=True, capture_output=True)
subprocess.run('pkill -f GatewayServer', shell=True, capture_output=True)
time.sleep(2)

try:
    from pyspark import SparkContext
    if SparkContext._active_spark_context:
        SparkContext._active_spark_context.stop()
    SparkContext._active_spark_context = None
    SparkContext._gateway = None
except Exception:
    pass

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, DoubleType, IntegerType
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName('Immobilier') \
    .config('spark.driver.memory', '2g') \
    .config('spark.sql.shuffle.partitions', '4') \
    .config('spark.ui.enabled', 'false') \
    .config('spark.driver.maxResultSize', '1g') \
    .config('spark.memory.offHeap.enabled', 'true') \
    .config('spark.memory.offHeap.size', '512m') \
    .master('local[2]') \
    .getOrCreate()

spark.sparkContext.setLogLevel('ERROR')
DATA_DIR = os.getcwd()
print(f'Spark {spark.version} — OK')
print(f'Repertoire : {DATA_DIR}')

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/15 16:49:37 WARN Utils: Your hostname, codespaces-d4c346, resolves to a loopback address: 127.0.0.1; using 10.0.3.141 instead (on interface eth0)
26/04/15 16:49:37 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/15 16:49:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.1.1 — OK
Repertoire : /workspaces/cours-spark-2026/Projet_scrapping


## 2. Chargement

In [3]:
df = spark.read.csv(
    os.path.join(DATA_DIR, 'immobilier_predictions.csv'),
    header=True, inferSchema=True
)
print(f'Lignes chargees : {df.count()}')
df.show(5, truncate=40)

Lignes chargees : 938
+---------------------+-----------+---------+----------------+------+-----------+-------+---------+---------+-----------+-------+--------------+
|                ville|code_postal|type_bien|type_transaction|source|departement|surface|     prix|nb_pieces|nb_chambres|prix_m2|prix_m2_predit|
+---------------------+-----------+---------+----------------+------+-----------+-------+---------+---------+-----------+-------+--------------+
|               Hyères|      83400|   Maison|           vente|  orpi|         83|  165.0|1390000.0|      9.0|        1.0|8424.24|       5085.22|
|            Lamorlaye|      60260|   Maison|           vente|   pap|         60|  289.0|1295000.0|     10.0|        7.0|4480.97|       2787.61|
|              Garches|      92380|   Maison|           vente|   pap|         92|  150.0|1090000.0|      3.0|        4.0|7266.67|       6232.15|
|Saint-maur-des-fossés|      94100|   Maison|           vente|   pap|         94|  151.0| 990000.0|      8.0

## 3. Prix au m² par ville et par type de bien

In [4]:
print('=== Top 20 villes — Prix/m2 moyen (vente) ===')
df.filter(F.col('type_transaction') == 'vente') \
  .groupBy('ville') \
  .agg(F.count('*').alias('nb'), F.round(F.mean('prix_m2'), 0).alias('prix_m2_moyen')) \
  .filter(F.col('nb') >= 5) \
  .orderBy(F.desc('prix_m2_moyen')) \
  .show(20, truncate=False)

print('\n=== Prix/m2 par type de bien et transaction ===')
df.groupBy('type_bien', 'type_transaction') \
  .agg(F.count('*').alias('nb'), F.round(F.mean('prix_m2'), 0).alias('prix_m2_moyen')) \
  .orderBy('type_bien', 'type_transaction').show()

=== Top 20 villes — Prix/m2 moyen (vente) ===
+-------------------+---+-------------+
|ville              |nb |prix_m2_moyen|
+-------------------+---+-------------+
|Paris              |166|10821.0      |
|Saint-cloud        |6  |8165.0       |
|Garches            |8  |6542.0       |
|Rueil-malmaison    |8  |6528.0       |
|Maisons-alfort     |8  |6484.0       |
|Châtenay-malabry   |5  |5433.0       |
|Marseille          |12 |5387.0       |
|Antony             |5  |4879.0       |
|Lyon               |5  |4846.0       |
|Haÿ-les-roses      |11 |4808.0       |
|Villejuif          |6  |4762.0       |
|Neuilly-plaisance  |11 |4497.0       |
|Champigny-sur-marne|12 |4193.0       |
|Épinay-sur-seine   |5  |4051.0       |
|Vitry-sur-seine    |10 |4029.0       |
|Lille              |9  |4015.0       |
|Strasbourg         |5  |4008.0       |
|Argenteuil         |9  |3867.0       |
|Créteil            |6  |3790.0       |
|Toulouse           |9  |3629.0       |
+-------------------+---+---------

## 4. Rendement locatif brut

`rendement = (loyer_mensuel × 12) / prix_achat × 100`

In [5]:
# Rendement locatif brut
vente_df = df.filter(F.col('type_transaction') == 'vente') \
    .groupBy('ville') \
    .agg(F.count('*').alias('nb_ventes'), F.mean('prix').alias('prix_vente_moyen')) \
    .filter(F.col('nb_ventes') >= 3)

location_df = df.filter(F.col('type_transaction') == 'location') \
    .groupBy('ville') \
    .agg(F.count('*').alias('nb_locations'), F.mean('prix').alias('loyer_moyen')) \
    .filter(F.col('nb_locations') >= 3)

vente_df.join(location_df, on='ville', how='inner') \
    .withColumn('rendement_brut_pct',
        F.round((F.col('loyer_moyen') * 12) / F.col('prix_vente_moyen') * 100, 2)) \
    .filter(F.col('rendement_brut_pct') > 0) \
    .orderBy(F.desc('rendement_brut_pct')) \
    .select('ville','nb_ventes','nb_locations',
            F.round('prix_vente_moyen',0).alias('prix_vente_moyen'),
            F.round('loyer_moyen',0).alias('loyer_moyen'),
            'rendement_brut_pct') \
    .show(15, truncate=False)

+-----+---------+------------+----------------+-----------+------------------+
|ville|nb_ventes|nb_locations|prix_vente_moyen|loyer_moyen|rendement_brut_pct|
+-----+---------+------------+----------------+-----------+------------------+
+-----+---------+------------+----------------+-----------+------------------+



## 5. Scoring des biens

In [6]:
df_scored = df \
    .withColumn('ecart_pct',
        F.round((F.col('prix_m2') - F.col('prix_m2_predit')) / F.col('prix_m2_predit') * 100, 2)) \
    .withColumn('score_marche',
        F.when(F.col('ecart_pct') < -15, 'Sous-evalue')
         .when(F.col('ecart_pct') >  15, 'Sur-evalue')
         .otherwise('Prix marche'))

print('=== Distribution ===')
df_scored.groupBy('score_marche').count().orderBy('count').show()

df_scored.select(
    'ville','type_bien',
    F.round('surface',1).alias('surface'),
    F.round('prix_m2',2).alias('prix_m2_reel'),
    'prix_m2_predit','ecart_pct','score_marche'
).show(10, truncate=40)

=== Distribution ===
+------------+-----+
|score_marche|count|
+------------+-----+
|  Sur-evalue|  245|
| Sous-evalue|  329|
| Prix marche|  364|
+------------+-----+

+---------------------+---------+-------+------------+--------------+---------+------------+
|                ville|type_bien|surface|prix_m2_reel|prix_m2_predit|ecart_pct|score_marche|
+---------------------+---------+-------+------------+--------------+---------+------------+
|               Hyères|   Maison|  165.0|     8424.24|       5085.22|    65.66|  Sur-evalue|
|            Lamorlaye|   Maison|  289.0|     4480.97|       2787.61|    60.75|  Sur-evalue|
|              Garches|   Maison|  150.0|     7266.67|       6232.15|     16.6|  Sur-evalue|
|Saint-maur-des-fossés|   Maison|  151.0|     6556.29|       4776.37|    37.27|  Sur-evalue|
|           Courbevoie|   Maison|  120.0|     7908.33|       6296.37|     25.6|  Sur-evalue|
|              Garches|   Maison|  150.0|     6266.67|       6232.15|     0.55| Prix ma

## 6. Top 15 meilleures affaires & biens sur-évalués

In [7]:
cols = ['ville','type_bien','type_transaction',
        F.round('surface',1).alias('surface'),
        F.round('prix',0).alias('prix'),
        F.round('prix_m2',2).alias('prix_m2_reel'),
        'prix_m2_predit','ecart_pct','score_marche']

print('=== Top 15 meilleures affaires ===')
df_scored.filter(F.col('score_marche') == 'Sous-evalue') \
         .orderBy(F.asc('ecart_pct')).select(*cols).show(15, truncate=30)

print('\n=== Top 15 biens sur-evalues ===')
df_scored.filter(F.col('score_marche') == 'Sur-evalue') \
         .orderBy(F.desc('ecart_pct')).select(*cols).show(15, truncate=30)

=== Top 15 meilleures affaires ===
+-------------------+-----------+----------------+-------+--------+------------+--------------+---------+------------+
|              ville|  type_bien|type_transaction|surface|    prix|prix_m2_reel|prix_m2_predit|ecart_pct|score_marche|
+-------------------+-----------+----------------+-------+--------+------------+--------------+---------+------------+
|            Tanques|     Maison|           vente|   87.0| 44000.0|      505.75|       3164.58|   -84.02| Sous-evalue|
|           Boulogne|    inconnu|           vente|   14.0| 10900.0|      778.57|       3563.06|   -78.15| Sous-evalue|
|          Randonnai|     Maison|           vente|   75.0| 70000.0|      933.33|       3458.32|   -73.01| Sous-evalue|
|            Limoges|     Maison|           vente|  208.0|195000.0|       937.5|       2787.61|   -66.37| Sous-evalue|
|              Saint|Appartement|           vente|  118.0|126900.0|     1075.42|       3164.58|   -66.02| Sous-evalue|
|           J

## 7. Scoring par ville

In [8]:
print('=== Scoring par ville (min 5 annonces) ===')
df_scored.groupBy('ville') \
    .agg(
        F.count('*').alias('nb_biens'),
        F.round(F.mean('prix_m2'), 0).alias('prix_m2_moyen'),
        F.round(F.sum(F.when(F.col('score_marche')=='Sous-evalue',1).otherwise(0)) /
                F.count('*') * 100, 1).alias('pct_sous_evalues'),
        F.round(F.sum(F.when(F.col('score_marche')=='Sur-evalue',1).otherwise(0)) /
                F.count('*') * 100, 1).alias('pct_sur_evalues')
    ) \
    .filter(F.col('nb_biens') >= 5) \
    .orderBy(F.desc('pct_sous_evalues')) \
    .show(20, truncate=False)

=== Scoring par ville (min 5 annonces) ===
+-------------------+--------+-------------+----------------+---------------+
|ville              |nb_biens|prix_m2_moyen|pct_sous_evalues|pct_sur_evalues|
+-------------------+--------+-------------+----------------+---------------+
|Tourcoing          |5       |1999.0       |100.0           |0.0            |
|Boulogne           |11      |2052.0       |81.8            |9.1            |
|Créteil            |6       |3790.0       |66.7            |0.0            |
|Vitry-sur-seine    |10      |4029.0       |60.0            |0.0            |
|Antony             |5       |4879.0       |60.0            |0.0            |
|Châtenay-malabry   |5       |5433.0       |60.0            |0.0            |
|Champigny-sur-marne|12      |4193.0       |58.3            |8.3            |
|Rosny-sous-bois    |6       |3489.0       |50.0            |16.7           |
|Saint              |28      |2931.0       |46.4            |10.7           |
|Toulouse           |

## 8. Export final

In [9]:
out = os.path.join(DATA_DIR, 'immobilier_scored.csv')
if os.path.exists(out):
    shutil.rmtree(out) if os.path.isdir(out) else os.remove(out)
df_scored.select(
    'ville', 'code_postal', 'type_bien', 'type_transaction', 'source',
    F.round('surface', 1).alias('surface'),
    F.round('prix', 0).alias('prix'),
    'nb_pieces', 'nb_chambres',
    F.round('prix_m2', 2).alias('prix_m2'),
    'prix_m2_predit', 'ecart_pct', 'score_marche'
).toPandas().to_csv(out, index=False, encoding='utf-8')
print(f'immobilier_scored.csv exporte : {df_scored.count()} lignes')

immobilier_scored.csv exporte : 938 lignes


In [ ]:
spark.stop()
print('Spark termine.')